# 8-2절 연습 문제 풀이

이 노트북은 8-2절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch08/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 8장 공통 - MiniVGGNet / 배치 정규화 / 잔차 블록
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import torchinfo
DATA_ROOT = '../../download'

def cifar_loaders(batch_size=64, train_transform=None):
    tf = transforms.Compose([transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))])
    full = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True,
                            transform=train_transform or tf)
    test = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True, transform=tf)
    g = torch.Generator().manual_seed(SEED)
    n_val = int(len(full) * 0.2)
    tr, va = random_split(full, [len(full) - n_val, n_val], generator=g)
    return (DataLoader(tr, batch_size=batch_size, shuffle=True),
            DataLoader(va, batch_size=batch_size), DataLoader(test, batch_size=batch_size))

def vgg_block(fan_in, fan_out, n_conv=2, bn=False):
    layers = []
    for i in range(n_conv):
        layers.append(nn.Conv2d(fan_in if i == 0 else fan_out, fan_out, 3, 1, 1))
        if bn: layers.append(nn.BatchNorm2d(fan_out))
        layers.append(nn.ReLU())
    layers.append(nn.MaxPool2d(2))
    return nn.Sequential(*layers)

def make_vgg(channels=(32, 64, 128), n_conv=2, bn=False, dropout=0.0):
    blocks, fan_in, size = [], 3, 32
    for ch in channels:
        blocks.append(vgg_block(fan_in, ch, n_conv, bn)); fan_in, size = ch, size // 2
    head = [nn.Flatten()]
    if dropout: head.append(nn.Dropout(dropout))
    head.append(nn.Linear(fan_in * size * size, 10))
    return nn.Sequential(*blocks, *head)

def fit(model, epochs=10, lr=1e-3, loaders=None):
    tr, va, te = loaders or cifar_loaders()
    model = model.to(device)
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for e in range(1, epochs + 1):
        model.train()
        for x, y in tr:
            loss = crit(model(x.to(device)), y.to(device))
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval(); c = n = 0
        with torch.no_grad():
            for x, y in va:
                c += (model(x.to(device)).argmax(1).cpu() == y).sum().item(); n += len(y)
        print(f'  {e}/{epochs} 검증 정확도 {c / n * 100:.2f}%')
    model.eval(); c = n = 0
    with torch.no_grad():
        for x, y in te:
            c += (model(x.to(device)).argmax(1).cpu() == y).sum().item(); n += len(y)
    print(f'  평가 정확도 {c / n * 100:.2f}%')
    return c / n * 100

class VGGAutoEncoder(nn.Module):
    def __init__(self, bn=False):
        super().__init__()
        def enc_block(i, o):
            layers = [nn.Conv2d(i, o, 3, 1, 1)]
            if bn: layers.append(nn.BatchNorm2d(o))
            layers += [nn.ReLU(), nn.Conv2d(o, o, 3, 1, 1)]
            if bn: layers.append(nn.BatchNorm2d(o))
            layers += [nn.ReLU(), nn.MaxPool2d(2)]
            return nn.Sequential(*layers)
        def dec_block(i, o):
            layers = [nn.ConvTranspose2d(i, o, 3, 2, 1, output_padding=1)]
            if bn: layers.append(nn.BatchNorm2d(o))
            layers += [nn.ReLU(), nn.Conv2d(o, o, 3, 1, 1)]
            if bn: layers.append(nn.BatchNorm2d(o))
            layers.append(nn.ReLU())
            return nn.Sequential(*layers)
        self.encoder = nn.Sequential(enc_block(3, 32), enc_block(32, 64))
        self.decoder = nn.Sequential(dec_block(64, 32), dec_block(32, 16),
                                     nn.Conv2d(16, 3, 3, 1, 1), nn.Sigmoid())
    def forward(self, x): return self.decoder(self.encoder(x))

## 연습 8-4

MiniVGGNetBN 모델에서 일부 배치 정규화 계층만 affine=False로 설정했을 때 학습 가능한 파라미터의 수가 어떻게 바뀌는지 torchinfo.summary()를 사용해 확인해 보자.

In [ ]:
class PartialAffineVGG(nn.Module):
    def __init__(self, affine_flags=(True, True, True)):
        super().__init__()
        blocks, fan_in = [], 3
        for ch, af in zip((32, 64, 128), affine_flags):
            blocks.append(nn.Sequential(
                nn.Conv2d(fan_in, ch, 3, 1, 1), nn.BatchNorm2d(ch, affine=af), nn.ReLU(),
                nn.Conv2d(ch, ch, 3, 1, 1), nn.BatchNorm2d(ch, affine=af), nn.ReLU(),
                nn.MaxPool2d(2)))
            fan_in = ch
        self.f = nn.Sequential(*blocks)
        self.c = nn.Sequential(nn.Flatten(), nn.Linear(128 * 4 * 4, 10))
    def forward(self, x): return self.c(self.f(x))

for flags in [(True, True, True), (False, True, True), (False, False, False)]:
    model = PartialAffineVGG(flags)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'affine={flags}: 전체 {total:,}개 / 학습 가능 {trainable:,}개')
torchinfo.summary(PartialAffineVGG((False, True, True)), input_size=(1, 3, 32, 32),
                  verbose=0)

`affine=True`인 배치 정규화 계층은 채널마다 크기(γ)와 이동(β) 파라미터를 학습하므로 **채널 수 × 2**만큼 파라미터가 생긴다. `affine=False`로 두면 정규화만 하고 이 파라미터가 사라진다. 첫 블록만 꺼도 32×2×2=128개가 줄어든다.

## 연습 8-5

MiniVGGNetBN 모델의 분류기에 앞서 삭제했던 드롭아웃 계층을 다시 추가해 학습해 보자. 학습 과정과 성능이 어떻게 달라지는지 관찰하고, 그 이유를 설명해 보자.

In [ ]:
for dr in (0.0, 0.5):
    torch.manual_seed(SEED)
    print(f'[드롭아웃 {dr}]')
    fit(make_vgg(bn=True, dropout=dr), epochs=10)
    print()

배치 정규화가 있는 모델에 드롭아웃을 더하면 대체로 **성능이 조금 떨어지거나 비슷**하다.

배치 정규화는 매 배치의 통계로 값을 다듬는 과정에서 이미 **약한 규제 효과**를 낸다. 여기에 드롭아웃까지 더하면 규제가 과해져 학습이 느려지고 과소적합으로 기운다. 또 드롭아웃이 만드는 분산 변화가 배치 정규화의 이동 통계와 충돌해 학습과 추론의 동작이 어긋나기도 한다. 그래서 최신 합성곱 모델은 드롭아웃을 잘 쓰지 않는다.

## 연습 8-6

배치 정규화 계층 덕분에 성능 저하 없이 더 깊은 신경망을 만들 수 있는지 MiniVGGNetBN 모델을 [연습 문제 8-2]처럼 다음 두 가지 방향으로 각각 수정해 직접 확인해 보자.

VGG 블록에 합성곱 계층, 배치 정규화 계층, ReLU 활성화 계층 한 묶음 더 추가한 모델

특징 추출기 계층(features)에 VGG 블록을 하나 더 추가한 모델

In [ ]:
configs = [('BN 원본', dict(channels=(32, 64, 128), n_conv=2, bn=True)),
           ('BN 블록당 합성곱 3개', dict(channels=(32, 64, 128), n_conv=3, bn=True)),
           ('BN 블록 4개', dict(channels=(32, 64, 128, 256), n_conv=2, bn=True))]
for name, kw in configs:
    torch.manual_seed(SEED)
    print(f'[{name}]')
    fit(make_vgg(**kw), epochs=10)
    print()

배치 정규화가 있으면 [연습 문제 8-2]와 달리 깊이를 늘려도 성능이 **떨어지지 않고 유지되거나 오른다**. 각 계층의 입력 분포가 안정되어 기울기가 앞쪽까지 잘 전달되기 때문이다. 다만 깊이만으로 계속 좋아지지는 않으며, 그 한계를 넘는 것이 잔차 연결이다.

## 연습 8-7

콜백 함수 data_inspect_hook()을 수정해 MiniVGGNetBN 모델에서 각 계층 출력 데이터의 최댓값과 최솟값을 출력해 보자. 그리고 출력에 가까운 계층일수록 이 값의 범위가 어떻게 변하는지 관찰해 보자.

In [ ]:
model = make_vgg(bn=True).to(device)
def range_hook(module, inputs, output):
    print(f'  {module.__class__.__name__:12s} 최소 {output.min().item():7.3f} '
          f'최대 {output.max().item():7.3f}')

handles = [m.register_forward_hook(range_hook) for m in model.modules()
           if isinstance(m, (nn.Conv2d, nn.BatchNorm2d, nn.ReLU))]
x = torch.randn(1, 3, 32, 32).to(device)
model.eval()
with torch.no_grad(): model(x)
for h in handles: h.remove()

출력에 가까운 계층일수록 값의 범위가 **넓어지는 경향**이 있다. 계층을 거칠 때마다 가중치가 곱해지며 값이 증폭되기 때문이다. 배치 정규화 계층 직후에는 범위가 다시 좁아지는데, 이것이 배치 정규화가 값의 폭주를 막아 주는 모습이다.

## 연습 8-8

CIFAR-10 샘플 하나에 해당하는 입력 텐서를 무작위 값으로 만들고 순전파와 역전파를 1회 실행하면서 계산된 MiniVGGNetBN 모델의 모든 합성곱 계층별 가중치 기울기의 평균을 출력해 보자.

힌트: 역전파 과정에서 실행되는 훅을 사용한다.

In [ ]:
model = make_vgg(bn=True).to(device)
grad_means = {}
def grad_hook(name):
    def hook(module, grad_input, grad_output):
        grad_means[name] = module.weight.grad.mean().item() if module.weight.grad is not None else None
    return hook

convs = [(n_, m) for n_, m in model.named_modules() if isinstance(m, nn.Conv2d)]
handles = [m.register_full_backward_hook(grad_hook(n_)) for n_, m in convs]

x = torch.randn(1, 3, 32, 32).to(device)
y = torch.tensor([3]).to(device)
loss = nn.CrossEntropyLoss()(model(x), y)
model.zero_grad(); loss.backward()
for h in handles: h.remove()

for n_, m in convs:
    g = m.weight.grad
    print(f'{n_:14s} 기울기 평균 {g.mean().item():+.3e} / 표준편차 {g.std().item():.3e}')

역방향 훅(`register_full_backward_hook`)은 역전파가 그 계층을 지날 때 호출된다. 다만 훅이 불릴 시점에는 `weight.grad`가 아직 갱신 전일 수 있어, 위 코드처럼 `backward()`가 끝난 뒤 `weight.grad`를 직접 확인하는 편이 안전하다.

입력에 가까운 계층일수록 기울기의 크기가 작아지는지 확인하면 기울기 소실 여부를 알 수 있다.

## 연습 8-9

[도전 문제] [연습 문제 8-3]에서 만든 VGGNet 기반의 오토인코더 모델을 수정해 배치 정규화 계층을 추가해 보자. 학습한 오토인코더가 배치 정규화 계층 유무에 따라 결과가 어떻게 다른지 비교하고, 그 결과를 배치 정규화 계층의 역할을 기준으로 설명해 보자.

In [ ]:
tf = transforms.ToTensor()
loader = DataLoader(datasets.CIFAR10(root=DATA_ROOT, train=True, download=True,
                                     transform=tf), batch_size=128, shuffle=True)
for bn in (False, True):
    torch.manual_seed(SEED)
    ae = VGGAutoEncoder(bn=bn).to(device)
    crit = nn.MSELoss(); opt = torch.optim.Adam(ae.parameters(), lr=1e-3)
    losses = []
    for e in range(5):
        ae.train(); tot = n = 0
        for x, _ in loader:
            x = x.to(device)
            loss = crit(ae(x), x)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(x); n += len(x)
        losses.append(tot / n)
    print(f'배치 정규화 {"있음" if bn else "없음"}: 손실 추이 '
          f'{[round(l, 5) for l in losses]}')

배치 정규화가 있으면 초기 손실이 더 빠르게 떨어지고 학습이 안정적이다. 오토인코더는 인코더와 디코더를 거치며 계층이 깊어지는데, 정규화가 각 단계의 값 범위를 유지해 주기 때문이다.

다만 출력이 0~1 범위여야 하는 복원 과제 특성상 **마지막 계층에는 배치 정규화를 넣지 않는다**(11장 DCGAN 생성자와 같은 원리).

## 연습 8-10

[도전 문제] 훅을 사용해 데이터를 관찰하는 대표적인 예제는 합성곱 신경망 모델에서 합성곱 계층이 출력하는 특징 지도를 계층별로 확인해 보는 것이다. MiniVGGNetBN 모델의 첫 번째, 세 번째, 마지막 합성곱 계층이 출력하는 특징 지도 중 각각 여덟 개씩 골라 이미지 파일에 저장해 보고, 특징 지도가 나타내는 바를 설명해 보자. 깃허브 노트북 예제의 save_feature_maps() 함수를 사용하면 특징 지도를 이미지 파일에 저장할 수 있다.

In [ ]:
model = make_vgg(bn=True).to(device)
feature_maps = {}
def save_hook(name):
    def hook(module, inputs, output): feature_maps[name] = output.detach().cpu()
    return hook

convs = [(n_, m) for n_, m in model.named_modules() if isinstance(m, nn.Conv2d)]
picks = [convs[0], convs[2], convs[-1]]     # 첫 번째, 세 번째, 마지막
handles = [m.register_forward_hook(save_hook(n_)) for n_, m in picks]

sample = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True,
                          transform=transforms.ToTensor())[0][0]
with torch.no_grad():
    model(sample.unsqueeze(0).to(device))
for h in handles: h.remove()

for name, fmap in feature_maps.items():
    print(f'{name}: {tuple(fmap.shape)}')
    viz.plot_feature_map(fmap[0][:8], title=f'{name} 특징 지도 8개', images_per_row=8)

**첫 번째 합성곱**은 경계·색 대비 같은 저수준 특징이라 원본 형태가 뚜렷이 남아 있다. **중간 계층**은 부분 패턴(질감, 부분 모양)을 담고, **마지막 합성곱**은 크기가 작고 추상적이어서 사람 눈으로는 무엇을 담았는지 알아보기 어렵다.

계층이 깊어질수록 '보이는 것'에서 '의미하는 것'으로 옮겨 가는 과정이 시각적으로 확인된다.